In [1]:
import re




def normalize_model(text: str) -> str:
    if not text or not isinstance(text, str):
        return text

    # 1. Верхний регистр
    text = text.upper()

    # 2. Удаление всех непечатных и невидимых символов
    text = re.sub(r'[\x00-\x1f\x7f\xa0\u200B-\u200D\uFEFF]', '', text)

    # Замена всех видов дефисов, минусов и тире на стандартный дефис (U+002D)
    text = re.sub(r'[\u2010-\u2015\u2212\uFE63\uFF0D]', '-', text)

    # 7. Обработка №… (вырезаем, добавим в конце)
    numbers = re.findall(r'№\d+', text)
    if numbers:
        text = re.sub(r'№\d+', '', text)

    # 6. Запятые между цифрами -> точки
    text = re.sub(r'(\d),(\d)', r'\1.\2', text)

    # 8. Латинская X между цифрами -> дефис
    text = re.sub(r'(\d)X(\d)', r'\1-\2', text)

    # Замена всех разделителей (пробел, _, /, *, \) на дефис
    text = text.replace(' ', '-')
    for ch in '_/*\\':
        text = text.replace(ch, '-')

    # 12. Римская I: греческая йота -> латинская I
    text = text.replace('Ι', 'I')

    # 3. Удаляем ВСЕ дефисы между буквой и цифрой (и наоборот)
    text = re.sub(r'([А-ЯA-Z])-+(\d)', r'\1\2', text)
    text = re.sub(r'(\d)-+([А-ЯA-Z])', r'\1\2', text)

    # Убираем двойные и более дефисы, дефис по краям
    text = re.sub(r'-+', '-', text)
    text = text.strip('-')

    # 9. Ё -> Е
    text = text.replace('Ё', 'Е')

    # 10. Замена 11 запрещённых латинских букв на кириллицу
    forbidden = {
        'A': 'А', 'B': 'В', 'C': 'С', 'E': 'Е', 'H': 'Н',
        'K': 'К', 'M': 'М', 'O': 'О', 'P': 'Р', 'T': 'Т', 'X': 'Х'
    }
    text = ''.join(forbidden.get(c, c) for c in text)

    # 7. Восстановление (№...)
    if numbers:
        text = text.rstrip('-') + f'({numbers[-1]})'

    return text


# ---------------------- ВАЛИДАЦИЯ ----------------------
def validate_model(model: str) -> list:
    errors = []

    # Правило 1
    if any(c.islower() for c in model if c.isalpha()):
        errors.append("Есть строчные буквы (нарушение правила 1)")

    # Правило 2
    if ' ' in model:
        errors.append("Присутствуют пробелы (нарушение правила 2)")
    if re.search(r'[\x00-\x1f\x7f\xa0]', model):
        errors.append("Непечатные символы (нарушение правила 2)")

    # Правило 3
    if re.search(r'[А-ЯA-Z][^А-ЯA-Z0-9][0-9]', model):
        errors.append("Знак между буквой и цифрой (нарушение правила 3)")
    if re.search(r'[0-9][^А-ЯA-Z0-9][А-ЯA-Z]', model):
        errors.append("Знак между цифрой и буквой (нарушение правила 3)")

    # Правило 4
    if re.search(r'[А-ЯA-Z][^А-ЯA-Z0-9-][А-ЯA-Z]', model):
        errors.append("Недопустимый разделитель между буквами (нарушение правила 4)")

    # Правило 5 – ИСПРАВЛЕНО
    if re.search(r'[0-9]([^0-9.-]|--+)[0-9]', model):
        if not re.search(r'[0-9][A-ZА-Я][0-9]', model):
            errors.append("Недопустимый разделитель между цифрами (нарушение правила 5)")

    # Правило 6
    if re.search(r'[0-9],[0-9]', model):
        errors.append("Запятая между цифрами (нарушение правила 6)")

    # Правило 7
    if '№' in model and not re.search(r'\(№\d+\)$', model):
        errors.append("Символ № не в конце или не в скобках (нарушение правила 7)")

    # Правило 8
    if re.search(r'[0-9]X[0-9]', model):
        errors.append("Латинская X между цифрами (нарушение правила 8)")

    # Правило 9
    if 'Ё' in model:
        errors.append("Буква Ё (нарушение правила 9)")

    # Правило 10
    forbidden_letters = {'A','B','C','E','H','K','M','O','P','T','X'}
    if any(c in forbidden_letters for c in model):
        errors.append("Запрещённая латинская буква (нарушение правила 10)")

    # Греческая йота
    if 'Ι' in model:
        errors.append("Греческая буква Ι (должна быть заменена на I)")

    return errors


In [2]:
test = """
ПРГ-160;
GA 37-7,5P;
GA30-10;
T 19 V800-1;
T 19 V800-2;
MNSH 200;
ДК 112-80-2,2_220;
ДК 112-90-3,0_220;
ДКУ 112-120-3,0_110;
ДКУ 112-90-3,0__220;
S3CB-H070410-SS;
Д 160 112/а;
Д 200-36/Б;
1 Д200-90/а;
Д 160-112/Б;
MX -04A-PAC/1;
K37 R-AQA80/1;
R67 AQH80/1;
SEPREMIUM 30/RS;
UFS-SP30N;
UFS-SP120N;
MML 2550-M;
WT-MNT PREDOS;
PKF 2,0;
FMA 2004;
FMA 2006;
FMA 2012;
ARKAL 4*2;
WTMNTF 97-Ι"""

In [3]:
test1 = ["  ga  37 - 7 , 5 p  ",                     # пробелы, строчные, запятые
    "t_19_9_7___30",                           # много подчёркиваний
    "MNSH///200",                               # слеши
    "ДК****112---80***2,2____220",              # звёздочки, дефисы, подчёркивания
    "S3CB\\H070410\\SS",                        # обратные слеши
    "Д\160\112/а",                              # смесь слешей и пробелов
    " 1   Д200  90   а  ",                      # много пробелов
    "MX**-04A**PAC//1",                         # звёздочки и слеши
    "K37 R-AQA80/1 №123",                       # знак номера в конце
    "№456 WT-MNT PREDOS",                       # знак номера в начале
    "WTMNTF 97-Ι №789",                         # номер в конце
    "FMA 2004 (№111)",                          # уже есть скобки
    "  PKF  2 , 0  ",                           # пробелы и запятая
    "АRКАL  4  *  2",                           # пробелы и звёздочка
    "SEPREMIUM 30//RS",                         # двойной слеш
    "UFS--SP--30N",                             # двойные дефисы
    "MML  2550---M",                            # тройные дефисы
    "WT-MNT   PREDOS   EXTRA",                  # лишние слова
    "GA37-7,5P  (версия 2)",                    # скобки с текстом
    "Д160-112А   ",                             # пробелы в конце
    "  МХ04А-РАС1",                             # пробелы в начале
    "S3СВ-Н070410SS   ",                        # пробелы в конце
    "Д160112А  ",                               # пробелы
    "WТМNТF97I   ",                             # пробелы
    "1Д200-90А   ",                             # пробелы
    "FМА2004   ",                               # пробелы
    "АRКАL4-2   ",                              # пробелы
    "РКF2.0   ",                                # пробелы
    "GА30-10   ",                               # пробелы
    "1дД200‑90А",   # U+2010
    "1Д200−90А",   # U+2212
    "1Д200-\u200B90А",  # дефис + zero-width space
    "1Д200 -90А",
    "1Д200- 90А",
    "1_200-90А",
    "Д160‑112А",
    "Д160−112А",
    "Д160-\u200B112А",
    "Д160 -112А",
    "Д160- 112А"
]

In [4]:
for model in test.split(";"):
    result = normalize_model(model)
    print(result, end = "    ")
    errors = validate_model(result)
    print(errors)

ПРГ160    []
GА37-7.5Р    []
GА30-10    []
Т19V800-1    []
Т19V800-2    []
МNSН200    []
ДК112-80-2.2-220    []
ДК112-90-3.0-220    []
ДКУ112-120-3.0-110    []
ДКУ112-90-3.0-220    []
S3СВ-Н070410SS    []
Д160-112А    []
Д200-36Б    []
1Д200-90А    []
Д160-112Б    []
МХ04А-РАС1    []
К37R-АQА80-1    []
R67АQН80-1    []
SЕРRЕМIUМ30RS    []
UFS-SР30N    []
UFS-SР120N    []
ММL2550М    []
WТ-МNТ-РRЕDОS    []
РКF2.0    []
FМА2004    []
FМА2006    []
FМА2012    []
АRКАL4-2    []
WТМNТF97I    []


In [5]:
for model in test1:
    result = normalize_model(model)
    print(result, end = "    ")
    errors = validate_model(result)
    print(errors)

GА37-7-,-5Р    []
Т19-9-7-30    []
МNSН200    []
ДК112-80-2.2-220    []
S3СВ-Н070410SS    []
ДРJ-А    []
1Д200-90А    []
МХ04А-РАС1    []
К37R-АQА80-1(№123)    []
WТ-МNТ-РRЕDОS(№456)    []
WТМNТF97I(№789)    []
FМА2004-()(№111)    []
РКF2-,-0    []
АRКАL4-2    []
SЕРRЕМIUМ30RS    []
UFS-SР30N    []
ММL2550М    []
WТ-МNТ-РRЕDОS-ЕХТRА    []
GА37-7.5Р-(ВЕРСИЯ2)    []
Д160-112А    []
МХ04А-РАС1    []
S3СВ-Н070410SS    []
Д160112А    []
WТМNТF97I    []
1Д200-90А    []
FМА2004    []
АRКАL4-2    []
РКF2.0    []
GА30-10    []
1ДД200-90А    []
1Д200-90А    []
1Д200-90А    []
1Д200-90А    []
1Д200-90А    []
1-200-90А    []
Д160-112А    []
Д160-112А    []
Д160-112А    []
Д160-112А    []
Д160-112А    []


In [21]:
import pandas as pd

In [30]:
data = pd.read_excel("Нормализация моделей.xlsx")
data.head()

,Класс,Подкласс,Модель до нормализации,Модель после нормализации
0,Диспергаторы,Аппараты проточно-роторные,ПРГ-160,ПРГ160
1,Компрессоры,Винтовые,"GA 37-7,5P",GA37-7.5P
2,Компрессоры,Винтовые,GA30-10,GA30-10
3,Конвейеры,Ленточные,T 19 V800-1,T19V800-1
4,Конвейеры,Ленточные,T 19 V800-2,T19V800-2


In [41]:
data1 = data.copy()

for col in ["Модель до нормализации", "Модель после нормализации"]:
    # Получаем список значений столбца
    test = data1[col].tolist()
    # Вычисляем ошибки для каждой модели
    errors = [validate_model(model) for model in test]
    # Находим индекс проверяемого столбца1
    idx = data1.columns.get_loc(col) + 1  # позиция после него
    # Вставляем новый столбец с валидацией
    data1.insert(idx, f"{col}_validate", errors)
test= data1["Модель до нормализации"].tolist()
data1["func_result"] = [normalize_model(model) for model in test]
data1["func_result_validate"]=  [validate_model(model) for model in data1["func_result"].tolist()]
data1

,Класс,Подкласс,Модель до нормализации,Модель до нормализации_validate,Модель после нормализации,Модель после нормализации_validate,func_result,func_result_validate
0,Диспергаторы,Аппараты проточно-роторные,ПРГ-160,[Знак между буквой и цифрой (нарушение правила...,ПРГ160,[],ПРГ160,[]
1,Компрессоры,Винтовые,"GA 37-7,5P","[Присутствуют пробелы (нарушение правила 2), З...",GA37-7.5P,[Запрещённая латинская буква (нарушение правил...,GА37-7.5Р,[]
2,Компрессоры,Винтовые,GA30-10,[Запрещённая латинская буква (нарушение правил...,GA30-10,[Запрещённая латинская буква (нарушение правил...,GА30-10,[]
3,Конвейеры,Ленточные,T 19 V800-1,"[Присутствуют пробелы (нарушение правила 2), З...",T19V800-1,[Запрещённая латинская буква (нарушение правил...,Т19V800-1,[]
4,Конвейеры,Ленточные,T 19 V800-2,"[Присутствуют пробелы (нарушение правила 2), З...",T19V800-2,[Запрещённая латинская буква (нарушение правил...,Т19V800-2,[]
5,Конвейеры,Шнековые,MNSH 200,"[Присутствуют пробелы (нарушение правила 2), З...",MNSH200,[Запрещённая латинская буква (нарушение правил...,МNSН200,[]
6,Машины электрические,Коллекторные,"ДК 112-80-2,2_220","[Присутствуют пробелы (нарушение правила 2), З...",ДК112-80-2.2-220,[],ДК112-80-2.2-220,[]
7,Машины электрические,Коллекторные,"ДК 112-90-3,0_220","[Присутствуют пробелы (нарушение правила 2), З...",ДК112-90-3.0-220,[],ДК112-90-3.0-220,[]
8,Машины электрические,Коллекторные,"ДКУ 112-120-3,0_110","[Присутствуют пробелы (нарушение правила 2), З...",ДКУ112-120-3.0-110,[],ДКУ112-120-3.0-110,[]
9,Машины электрические,Коллекторные,"ДКУ 112-90-3,0__220","[Присутствуют пробелы (нарушение правила 2), З...",ДКУ112-90-3.0-220,[],ДКУ112-90-3.0-220,[]


In [42]:
data1.to_excel("Нормализация моделей(сравнение результатов).xlsx", index=False)